## Naive Bayes - Lentes de Contato



In [3]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB

# Conjunto de treino transcrito da tabela da atividade
dados = [
    ("infantil", "miopia", "nao", "reduzida", "nenhuma"),
    ("infantil", "miopia", "sim", "normal", "gelatinosa"),
    ("infantil", "hipermetropia", "nao", "normal", "gelatinosa"),
    ("infantil", "hipermetropia", "sim", "normal", "dura"),
    ("adolescente", "miopia", "nao", "reduzida", "gelatinosa"),
    ("adolescente", "miopia", "sim", "reduzida", "nenhuma"),
    ("adolescente", "miopia", "nao", "normal", "dura"),
    ("adolescente", "hipermetropia", "nao", "reduzida", "gelatinosa"),
    ("adolescente", "hipermetropia", "sim", "normal", "dura"),
    ("adulto", "miopia", "nao", "normal", "gelatinosa"),
    ("adulto", "miopia", "sim", "normal", "dura"),
    ("adulto", "miopia", "sim", "normal", "gelatinosa"),
    ("adulto", "hipermetropia", "nao", "reduzida", "nenhuma"),
    ("adulto", "hipermetropia", "sim", "normal", "gelatinosa"),
    ("adulto", "hipermetropia", "nao", "normal", "gelatinosa"),
]

colunas = ["idade", "diagnostico", "astigmatismo", "taxa_lacrimal", "tipo_lente"]
df = pd.DataFrame(dados, columns=colunas)

print("Conjunto de treino (15 exemplos):")
df

Conjunto de treino (15 exemplos):


,idade,diagnostico,astigmatismo,taxa_lacrimal,tipo_lente
0,infantil,miopia,nao,reduzida,nenhuma
1,infantil,miopia,sim,normal,gelatinosa
2,infantil,hipermetropia,nao,normal,gelatinosa
3,infantil,hipermetropia,sim,normal,dura
4,adolescente,miopia,nao,reduzida,gelatinosa
5,adolescente,miopia,sim,reduzida,nenhuma
6,adolescente,miopia,nao,normal,dura
7,adolescente,hipermetropia,nao,reduzida,gelatinosa
8,adolescente,hipermetropia,sim,normal,dura
9,adulto,miopia,nao,normal,gelatinosa


In [4]:
# 1) Tabela de contagens por classe e probabilidades suavizadas (Laplace alpha=1)
alpha = 1.0
atributos = ["idade", "diagnostico", "astigmatismo", "taxa_lacrimal"]


ordem_classes = ["nenhuma", "dura", "gelatinosa"]
ordem_valores = {
    "idade": ["infantil", "adolescente", "adulto"],
    "diagnostico": ["miopia", "hipermetropia"],
    "astigmatismo": ["sim", "nao"],
    "taxa_lacrimal": ["reduzida", "normal"],
}
classes = [c for c in ordem_classes if c in df["tipo_lente"].unique()]


N = len(df)
K = len(classes)
contagem_classes = df["tipo_lente"].value_counts().reindex(classes, fill_value=0)
tabela_classes = pd.DataFrame(
    {
        "contagem_frac": [f"{int(n)}/{N}" for n in contagem_classes.values],
        "prior_laplace_frac": [f"{int(n + alpha)}/{int(N + alpha * K)}" for n in contagem_classes.values],
    },
    index=classes,
)

print("Contagem das classes e priors suavizados P(classe):")
display(tabela_classes)

for att in atributos:
    valores = ordem_valores[att]
    tabela_contagem = pd.DataFrame(index=valores, columns=classes)
    tabela_laplace_frac = pd.DataFrame(index=valores, columns=classes)
    tabela_laplace_prob = pd.DataFrame(index=valores, columns=classes)

    for c in classes:
        subset = df[df["tipo_lente"] == c]
        Nc = len(subset)
        V = len(valores)

        for v in valores:
            count = int((subset[att] == v).sum())
            prob_laplace = (count + alpha) / (Nc + alpha * V)

            tabela_contagem.loc[v, c] = f"{count}/{Nc}"
            tabela_laplace_frac.loc[v, c] = f"{int(count + alpha)}/{int(Nc + alpha * V)}"
            tabela_laplace_prob.loc[v, c] = round(float(prob_laplace), 6)

    print(f"\nAtributo: {att}")
    print("Contagens:")
    display(tabela_contagem)
    print("Laplace (frações):")
    display(tabela_laplace_frac)
    print("Laplace (probabilidades):")
    display(tabela_laplace_prob)

Contagem das classes e priors suavizados P(classe):


,contagem_frac,prior_laplace_frac
nenhuma,3/15,4/18
dura,4/15,5/18
gelatinosa,8/15,9/18



Atributo: idade
Contagens:


,nenhuma,dura,gelatinosa
infantil,1/3,1/4,2/8
adolescente,1/3,2/4,2/8
adulto,1/3,1/4,4/8


Laplace (frações):


...

,nenhuma,dura,gelatinosa
reduzida,3/3,0/4,2/8
normal,0/3,4/4,6/8


Laplace (frações):


,nenhuma,dura,gelatinosa
reduzida,4/5,1/6,3/10
normal,1/5,5/6,7/10


Laplace (probabilidades):


,nenhuma,dura,gelatinosa
reduzida,0.8,0.166667,0.3
normal,0.2,0.833333,0.7


In [5]:
# 2) Treino do Naive Bayes categórico no sklearn e previsão do paciente
atributos = ["idade", "diagnostico", "astigmatismo", "taxa_lacrimal"]
ordem_classes = ["nenhuma", "dura", "gelatinosa"]
ordem_valores = {
    "idade": ["infantil", "adolescente", "adulto"],
    "diagnostico": ["miopia", "hipermetropia"],
    "astigmatismo": ["sim", "nao"],
    "taxa_lacrimal": ["reduzida", "normal"],
}

X = df[atributos]
y = df["tipo_lente"]


categorias_encoder = [ordem_valores[a] for a in atributos]
encoder = OrdinalEncoder(
    categories=categorias_encoder,
    handle_unknown="use_encoded_value",
    unknown_value=-1,
    dtype=int,
 )
X_enc = encoder.fit_transform(X).astype(int)


N = len(y)
K = len(ordem_classes)
contagem_classes = y.value_counts().reindex(ordem_classes, fill_value=0)
class_prior_planilha = ((contagem_classes + 1) / (N + K)).values


classe_para_codigo = {c: i for i, c in enumerate(ordem_classes)}
codigo_para_classe = {i: c for c, i in classe_para_codigo.items()}
y_enc = y.map(classe_para_codigo).values

nb = CategoricalNB(alpha=1.0, class_prior=class_prior_planilha, fit_prior=True)
nb.fit(X_enc, y_enc)


paciente = pd.DataFrame(
    [{
        "idade": "infantil",
        "diagnostico": "hipermetropia",
        "astigmatismo": "nao",
        "taxa_lacrimal": "reduzida",
    }]
 )

paciente_enc = encoder.transform(paciente).astype(int)
proba = nb.predict_proba(paciente_enc)[0]
pred_cod = int(nb.predict(paciente_enc)[0])
pred = codigo_para_classe[pred_cod]

resultado = pd.DataFrame(
    {
        "classe": ordem_classes,
        "prob_normalizada": proba,
    }
).sort_values("prob_normalizada", ascending=False)

print("Paciente:")
display(paciente)
print(f"Previsao (classe mais provavel): {pred}")
print("\nProbabilidades normalizadas por classe:")
display(resultado)
print("\nResumo numerico:")
print(resultado.to_string(index=False))

Paciente:


,idade,diagnostico,astigmatismo,taxa_lacrimal
0,infantil,hipermetropia,nao,reduzida


Previsao (classe mais provavel): nenhuma

Probabilidades normalizadas por classe:


,classe,prob_normalizada
0,nenhuma,0.495556
2,gelatinosa,0.427628
1,dura,0.076816



Resumo numerico:
    classe  prob_normalizada
   nenhuma          0.495556
gelatinosa          0.427628
      dura          0.076816
